In [1]:
!pip install nltk -q

In [2]:
import nltk
nltk.download('punkt')

from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
corpus = """
natural language processing is very interesting
natural language processing helps computers understand text
machine learning is a part of artificial intelligence
artificial intelligence is transforming the world
language models predict the next word in a sentence
"""

In [4]:
def build_ngram_model(text, n):
    tokens = word_tokenize(text.lower())
    model = defaultdict(Counter)

    for gram in ngrams(tokens, n):
        prefix = gram[:-1]
        next_word = gram[-1]
        model[prefix][next_word] += 1

    return model

In [6]:
import nltk
nltk.download('punkt_tab') # Download the missing resource

bigram_model = build_ngram_model(corpus, 2)
trigram_model = build_ngram_model(corpus, 3)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
def predict_next_word(model, text, n):
    tokens = word_tokenize(text.lower())

    if len(tokens) < n-1:
        return "Not enough context"

    context = tuple(tokens[-(n-1):])

    if context in model:
        return model[context].most_common(3)  # top 3 suggestions
    else:
        return "No prediction found"

In [8]:
print(predict_next_word(bigram_model, "natural language", 2))
print(predict_next_word(trigram_model, "natural language processing", 3))
print(predict_next_word(trigram_model, "artificial intelligence is", 3))

[('processing', 2), ('models', 1)]
[('is', 1), ('helps', 1)]
[('transforming', 1)]


In [9]:
def autocomplete(text):
    # Try trigram first
    result = predict_next_word(trigram_model, text, 3)

    if result == "No prediction found":
        result = predict_next_word(bigram_model, text, 2)

    return result

In [10]:
print(autocomplete("natural language processing"))
print(autocomplete("machine learning"))
print(autocomplete("language models"))

[('is', 1), ('helps', 1)]
[('is', 1)]
[('predict', 1)]


In [11]:
def suggest_words(text):
    predictions = autocomplete(text)

    if isinstance(predictions, str):
        return predictions

    return [word for word, _ in predictions]

# Test
print(suggest_words("natural language"))

['processing']


In [12]:
def predict_with_prob(model, text, n):
    tokens = word_tokenize(text.lower())
    context = tuple(tokens[-(n-1):])

    if context not in model:
        return "No prediction"

    total = sum(model[context].values())

    probs = {
        word: count / total
        for word, count in model[context].items()
    }

    return sorted(probs.items(), key=lambda x: x[1], reverse=True)